# Laboratori 6 · Ushtrime Shtesë — Renditja topologjike dhe komponentet e lidhura fort, në kod

> Laboratori kryesor i shpjegoi algoritmet e Kahn-it dhe Kosaraju-t me gjurmime me dorë, dhe detyra e shtëpisë i
> zgjidh me arritshmëri, çift pas çifti. Këtu i shkruani algoritmet **lineare**, $O(V + E)$, funksion pas
> funksioni, dhe çdo qelizë `kontrollo(...)` ju tregon nëse janë të sakta. Nuk kërkon Pygame.

| # | Ushtrimi | Çfarë mësoni |
|:-:|---|---|
| 1 | A është renditje topologjike? | çfarë do të thotë "e saktë" |
| 2 | Algoritmi i Kahn-it | fuqitë hyrëse, radha, zbulimi i ciklit |
| 3 | Renditja nga DFS | rendi i kundërt i kohëve të mbarimit |
| 4 | Sa renditje ka? | renditja topologjike rrallë është e vetme |
| 5 | Grafi i transpozuar | brinjët e përmbysura |
| 6 | Algoritmi i Kosaraju-t | dy DFS, komponentet e lidhura fort |
| 7 | Grafi i kondensuar | KLF-të si kulme: gjithmonë një DAG |

## Kujtues teorik

**Paraqitja.** Një graf i drejtuar jepet si listë fqinjësie me **harqet dalëse**: `G = {"a": ["b"], ...}`.
Kulmet ndiqen në rendin e çelësave të `G`, fqinjët në rendin e listës.

**Renditja topologjike** e një DAG-u: një renditje e të gjitha kulmeve ku për çdo hark $u \to v$, $u$ vjen para $v$.
Ekziston **atëherë dhe vetëm atëherë** kur grafi nuk ka cikël.

| Algoritmi | Ideja | Koha |
|---|---|---|
| **Kahn** | nxirr një kulm me fuqi hyrëse 0; ul fuqitë e pasardhësve; përsërit | $O(V + E)$ |
| **DFS** | një kulm mbaron vetëm pasi kanë mbaruar të gjithë pasardhësit e tij | $O(V + E)$ |
| **Kosaraju** | DFS mbi $G$ për kohët e mbarimit, pastaj DFS mbi $G^T$ në rendin zbritës të tyre | $O(V + E)$ |

**Pse DFS jep një renditje.** Në një DAG, për çdo hark $u \to v$: $f[v] < f[u]$. Prandaj kulmet në rendin
**zbritës** të kohës së mbarimit janë një renditje topologjike.

**Pse funksionon Kosaraju.** $G$ dhe $G^T$ kanë të njëjtat KLF. Kulmi që mbaron i fundit në $G$ është në një KLF
*burim* të grafit të kondensuar; në $G^T$ ajo bëhet *grykë*, prandaj DFS-ja prej tij mbledh pikërisht KLF-në e vet.

In [1]:
# ── Mjeti i kontrollit ──────────────────────────────────────────────
# Ekzekutojeni këtë qelizë një herë, në fillim. Pas çdo ushtrimi, qeliza
# "kontrollo(...)" teston funksionin tuaj dhe ju tregon nëse është gati.

def kontrollo(titulli, testet):
    try:
        testet()
    except NotImplementedError:
        print(f"⏳ {titulli}: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.")
    except AssertionError as e:
        print(f"✗ {titulli}: {e}")
    except Exception as e:
        print(f"✗ {titulli}: kodi u ndal me gabim  {type(e).__name__}: {e}")
    else:
        print(f"✓ {titulli}: të gjitha testet kaluan!")


import random
from collections import deque
from itertools import permutations


def nga_harqet(V, A):
    # Lista e fqinjësisë së një grafi të drejtuar: kulmet në rendin e V, fqinjët në rend alfabetik.
    G = {v: [] for v in V}
    for u, v in A:
        if v not in G[u]:
            G[u].append(v)
    return {v: sorted(fq) for v, fq in G.items()}


def dag_i_rastit(rng, n, p=0.35):
    # Harqe vetëm nga një kulm i hershëm te një i mëvonshëm, me emrat e përzier: një DAG.
    emrat = [chr(97 + i) for i in range(n)]
    rng.shuffle(emrat)
    return nga_harqet(sorted(emrat), [(emrat[i], emrat[j]) for i in range(n) for j in range(i + 1, n) if rng.random() < p])


# Grafi i lëndëve nga Laboratori 6, kulmet në rendin që hyjnë në radhë
LENDET = {
    "MAT1": ["MAT2", "PRG1"],
    "MAT2": ["ALG1"],
    "PRG1": ["PRG2"],
    "ALG1": ["DB"],
    "PRG2": ["OS", "NET"],
    "OS": ["DB"],
    "DB": ["NET"],
    "NET": [],
}

## Ushtrimi 1 — A është renditje topologjike?

Para se të gjenerojmë renditje, duhet të dimë si ta **kontrollojmë** një të tillë.

**Detyra:** `eshte_renditje_topologjike(G, renditja)` kthen `True` vetëm nëse `renditja` përmban **çdo kulm të `G`
saktësisht një herë** dhe për çdo hark $u \to v$, $u$ vjen para $v$.

In [2]:
def eshte_renditje_topologjike(G, renditja):
    # PLOTESO: një permutacion i kulmeve, dhe pozicioni[u] < pozicioni[v] për çdo hark
    raise NotImplementedError

In [3]:
def _testet():
    e_sakte = ["MAT1", "MAT2", "PRG1", "ALG1", "PRG2", "OS", "DB", "NET"]
    assert eshte_renditje_topologjike(LENDET, e_sakte) is True, "renditja e gjurmuar në laborator është e saktë"
    assert eshte_renditje_topologjike(LENDET, ["MAT1", "PRG1", "PRG2", "OS", "MAT2", "ALG1", "DB", "NET"]) is True
    assert eshte_renditje_topologjike(LENDET, ["MAT2", "MAT1", "PRG1", "ALG1", "PRG2", "OS", "DB", "NET"]) is False, \
        "MAT1 → MAT2 kërkon MAT1 para MAT2"
    assert eshte_renditje_topologjike(LENDET, e_sakte[:-1]) is False, "mungon NET: duhen të gjitha kulmet"
    assert eshte_renditje_topologjike(LENDET, e_sakte + ["NET"]) is False, "NET dy herë: çdo kulm vetëm një herë"
    assert eshte_renditje_topologjike({"a": ["b"], "b": ["a"]}, ["a", "b"]) is False, "një cikël nuk ka renditje"

kontrollo("Ushtrimi 1 — Kontrolli i një renditjeje", _testet)

⏳ Ushtrimi 1 — Kontrolli i një renditjeje: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Ushtrimi 2 — Algoritmi i Kahn-it

**Detyra:** `kahn(G)` kthen një renditje topologjike, ose `None` nëse grafi ka cikël.

1. Llogaritni fuqinë hyrëse të çdo kulmi.
2. Futni në radhë, **në rendin e kulmeve të `G`**, ata me fuqi hyrëse 0.
3. Nxirrni nga radha, shtojeni në renditje, dhe ulni fuqinë hyrëse të çdo pasardhësi (në rendin e listës);
   ai që arrin 0 hyn në fund të radhës.
4. Nëse renditja ka më pak kulme se `G`, kulmet që mbetën janë në një cikël: ktheni `None`.

In [4]:
def kahn(G):
    # PLOTESO: fuqitë hyrëse, një deque me ata me 0, dhe kontrolli i ciklit në fund
    raise NotImplementedError

In [5]:
def _testet():
    assert kahn(LENDET) == ["MAT1", "MAT2", "PRG1", "ALG1", "PRG2", "OS", "DB", "NET"], \
        f"radha e Kahn-it mbi lëndët, si në gjurmimin e laboratorit: {kahn(LENDET)}"
    assert kahn({"a": ["b"], "b": ["c"], "c": ["a"], "d": []}) is None, "a → b → c → a është cikël: None"
    rng = random.Random(6)
    for _ in range(200):
        G = dag_i_rastit(rng, rng.randint(1, 9))
        assert eshte_renditje_topologjike(G, kahn(G)), f"renditje e gabuar për {G}"
        a, b = rng.sample(sorted(G), 2) if len(G) > 1 else (None, None)
        if a and b and b in G[a]:
            me_cikel = {**G, b: G[b] + [a]}
            assert kahn(me_cikel) is None, "harku i shtuar mbyll një cikël: Kahn duhet ta zbulojë"

kontrollo("Ushtrimi 2 — Kahn", _testet)

⏳ Ushtrimi 2 — Kahn: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Ushtrimi 3 — Renditja nga DFS

**Detyra:** `dfs_renditje(G)` kthen kulmet e një DAG-u në rendin **zbritës** të kohës së mbarimit. Nisni DFS
nga kulmet në rendin e `G` (nga çdo kulm ende i pavizituar), vizitoni fqinjët sipas rendit të listës, dhe
shtoni një kulm në një listë **kur mbaron**; në fund përmbyseni listën.

In [6]:
def dfs_renditje(G):
    # PLOTESO: viziton(u) rekursiv; mbaruar.append(u) pas fqinjëve; ktheni mbaruar[::-1]
    raise NotImplementedError

In [7]:
def _testet():
    assert dfs_renditje(LENDET) == ["MAT1", "PRG1", "PRG2", "OS", "MAT2", "ALG1", "DB", "NET"], \
        f"DFS mbi lëndët: {dfs_renditje(LENDET)}"
    rng = random.Random(8)
    for _ in range(200):
        G = dag_i_rastit(rng, rng.randint(1, 9))
        assert eshte_renditje_topologjike(G, dfs_renditje(G)), f"renditje e gabuar për {G}"

kontrollo("Ushtrimi 3 — Renditja nga DFS", _testet)

⏳ Ushtrimi 3 — Renditja nga DFS: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


> **Pyetje:** Kahn dhe DFS japin dy renditje të ndryshme për lëndët. Cila është "e sakta"? Pse?

## Ushtrimi 4 — Sa renditje ka?

**Detyra:** `numero_renditjet(G)` kthen numrin e renditjeve topologjike të një DAG-u. Me rekursion: në çdo
hap, provoni secilin kulm që mund të vijë tani (pa paraardhës të pavendosur), vendoseni dhe numëroni vazhdimet.
Mjafton për grafe të vegjël — numri rritet shumë shpejt.

In [8]:
def numero_renditjet(G):
    # PLOTESO: rekursion mbi bashkësinë e kulmeve të vendosur
    raise NotImplementedError

In [9]:
def _testet():
    assert numero_renditjet({"a": ["b"], "b": ["c"], "c": []}) == 1, "një varg ka vetëm një renditje"
    assert numero_renditjet({v: [] for v in "abcd"}) == 24, "pa harqe: çdo permutacion, 4! = 24"
    assert numero_renditjet(LENDET) == 10, f"lëndët kanë 10 renditje, u morën {numero_renditjet(LENDET)}"
    rng = random.Random(2)
    for _ in range(40):
        G = dag_i_rastit(rng, rng.randint(1, 6))
        sakte = sum(eshte_renditje_topologjike(G, list(r)) for r in permutations(G))
        assert numero_renditjet(G) == sakte, f"për {G} ka {sakte} renditje"

kontrollo("Ushtrimi 4 — Numri i renditjeve", _testet)

⏳ Ushtrimi 4 — Numri i renditjeve: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Ushtrimi 5 — Grafi i transpozuar

**Detyra:** `transpozo(G)` kthen grafin $G^T$ me të njëjtat kulme dhe çdo hark të përmbysur: $u \to v$ bëhet
$v \to u$. Fqinjët në rend alfabetik.

In [10]:
def transpozo(G):
    # PLOTESO
    raise NotImplementedError

In [11]:
def _testet():
    assert transpozo({"a": ["b", "c"], "b": ["c"], "c": []}) == {"a": [], "b": ["a"], "c": ["a", "b"]}
    assert transpozo(transpozo(LENDET)) == {v: sorted(fq) for v, fq in LENDET.items()}, "(Gᵀ)ᵀ = G"
    assert sum(map(len, transpozo(LENDET).values())) == sum(map(len, LENDET.values())), "i njëjti numër harqesh"

kontrollo("Ushtrimi 5 — Grafi i transpozuar", _testet)

⏳ Ushtrimi 5 — Grafi i transpozuar: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Ushtrimi 6 — Algoritmi i Kosaraju-t

**Detyra:** `kosaraju(G)` kthen listën e komponenteve të lidhura fort, secila si `set`, në rendin që gjenden:

1. DFS mbi $G$ (kulmet në rendin e `G`, fqinjët sipas listës) dhe ruani kulmet **në rendin që mbarojnë**.
2. Ndërtoni $G^T$ me `transpozo`.
3. Kaloni kulmet në rendin **zbritës** të mbarimit; nga çdo kulm i pavizituar ende në kalimin e dytë, një DFS
   mbi $G^T$ mbledh një komponente.

In [12]:
SHEMBULLI = {"0": ["1"], "1": ["2", "3"], "2": ["0"], "3": ["4"], "4": ["3", "5"], "5": [], "6": ["7"], "7": ["6"]}


def kosaraju(G):
    # PLOTESO: dy kalime DFS; i dyti mbi transpozo(G) në rendin reversed(mbaruar)
    raise NotImplementedError

In [13]:
def _arrin(G, s):
    # Kulmet e arritshëm nga s, për kontrollin sipas përkufizimit.
    pare, stiva = {s}, [s]
    while stiva:
        for v in G[stiva.pop()]:
            if v not in pare:
                pare.add(v)
                stiva.append(v)
    return pare


def _testet():
    assert kosaraju(SHEMBULLI) == [{"6", "7"}, {"0", "1", "2"}, {"3", "4"}, {"5"}], \
        f"si në gjurmimin e laboratorit: {kosaraju(SHEMBULLI)}"
    assert kosaraju(LENDET) == [{v} for v in dfs_renditje(LENDET)], "në një DAG çdo kulm është KLF më vete"
    rng = random.Random(12)
    for _ in range(150):
        V = [chr(97 + i) for i in range(rng.randint(1, 8))]
        G = nga_harqet(V, [(u, v) for u in V for v in V if u != v and rng.random() < 0.2])
        K = kosaraju(G)
        assert sorted(v for k in K for v in k) == sorted(V), "çdo kulm në saktësisht një komponente"
        arrin = {v: _arrin(G, v) for v in V}
        for k in K:
            for u in k:
                assert {v for v in V if v in arrin[u] and u in arrin[v]} == k, \
                    f"komponentja e {u} duhet të jetë pikërisht kulmet që arrijnë dhe arrihen prej tij"

kontrollo("Ushtrimi 6 — Kosaraju", _testet)

⏳ Ushtrimi 6 — Kosaraju: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Ushtrimi 7 — Grafi i kondensuar

Nëse çdo KLF tkurret në një kulm të vetëm, harqet midis tyre formojnë **grafin e kondensuar**. Ai është gjithmonë
një DAG: një cikël midis dy KLF-ve do t'i bashkonte në një.

**Detyra:** `kondenso(G, komponentet)` kthen grafin e kondensuar si listë fqinjësie mbi indekset e komponenteve
(`0, 1, …`), pa lake dhe pa harqe të përsëritur, fqinjët në rend rritës.

In [14]:
def kondenso(G, komponentet):
    # PLOTESO: ku[v] = indeksi i komponentes; një hark i ↔ j për çdo u → v me ku[u] ≠ ku[v]
    raise NotImplementedError

In [15]:
def _testet():
    K = kosaraju(SHEMBULLI)
    assert kondenso(SHEMBULLI, K) == {0: [], 1: [2], 2: [3], 3: []}, \
        "{0,1,2} → {3,4} → {5}, dhe {6,7} pa harqe dalëse"
    rng = random.Random(13)
    for _ in range(100):
        V = [chr(97 + i) for i in range(rng.randint(1, 8))]
        G = nga_harqet(V, [(u, v) for u in V for v in V if u != v and rng.random() < 0.25])
        H = kondenso(G, kosaraju(G))
        assert kahn(H) is not None, "grafi i kondensuar duhet të jetë një DAG"
        # Kosaraju i gjen KLF-të në një renditje topologjike të grafit të kondensuar.
        assert eshte_renditje_topologjike(H, list(range(len(H)))), "harqet shkojnë nga një KLF e hershme te një e mëvonshme"

kontrollo("Ushtrimi 7 — Grafi i kondensuar", _testet)

⏳ Ushtrimi 7 — Grafi i kondensuar: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Punë në Klasë

**Problemi 1 — Kahn me dorë.** Për recetën më poshtë, plotësoni radhën dhe fuqitë hyrëse pas çdo nxjerrjeje.
Hyrja në radhë sipas rendit të listës: *uji, makaronat, kripa, zierja, salca, kulloja, përzierja, servirja*.

```
uji → zierja      makaronat → zierja    kripa → zierja
zierja → kulloja  kulloja → përzierja   salca → përzierja   përzierja → servirja
```

**Problemi 2 — Kosaraju me dorë.** Për grafin `SHEMBULLI`, shkruani kohët e mbarimit të kalimit të parë dhe
rendin në të cilin kalimi i dytë i gjen komponentet. Pse kalimi i dytë nuk "rrjedh" nga {0, 1, 2} te {3, 4}?

**Problemi 3 — Sa renditje?** Një DAG me 4 kulme ka harqet a → c dhe b → d. Sa renditje topologjike ka? Rendisini
të gjitha. Kontrollojeni me `numero_renditjet`.

## Pyetje Reflektuese

1. Kur Kahn mbaron me më pak kulme se grafi, pse kulmet që mbetën janë patjetër në një cikël (ose pas një cikli)?
2. Në një DAG, pse vlen $f[v] < f[u]$ për çdo hark $u \to v$? Shqyrtoni rastet kur DFS e zbulon $v$ para, gjatë dhe
   pas $u$-së.
3. Pse $G$ dhe $G^T$ kanë të njëjtat komponente të lidhura fort?
4. Qasja e detyrës së shtëpisë (arritshmëria çift pas çifti) kushton $O(V(V + E))$, Kosaraju $O(V + E)$. Për një graf
   me 10 000 kulme dhe 50 000 harqe, rreth sa herë më shpejt është Kosaraju?